# 03 Baseline Model

Training a Linear Regression as the baseline model using the cleaned .csv from Week 3.

Testing Month: May 2026

## Set Up & Load

- Similar imports as used in previous files
- data pulled from local machine, cleaned .csv file is not available in public github repository

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import glob

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

In [ ]:
TEST_MONTH = "2026-05" #most recent month 
TRAIN_MONTHS = 5 #tbd = preceding months

DATA = "../../data/cleaned_CRMLSSold202512_202605.csv" 
DATE = "CloseDate"
TARGET = "ClosePrice"

#display formatting:
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
sns.set_theme(style="whitegrid")

In [ ]:
# load data

df = pd.read_csv(DATA, parse_dates=[DATE])

print(f"Shape: {df.shape}")
print("Months:", sorted(df[DATE].dt.to_period("M").astype(str).unique()))
print(f"Date range: {df[DATE].min().date()} -> {df[DATE].max().date()}")

# sanity checking
assert df[TARGET].isna().sum() == 0, "Target has missing values"
assert (df[TARGET] > 0).all(), "Impossible price values in target"
assert df[DATE].isna().sum() == 0, "Date column has missing values"
assert df[DATE].dt.to_period("M").astype(str).max() == TEST_MONTH, (f"Latest month in data != TEST_MONTH ({TEST_MONTH})")
print("All sanity checks passed.")

## Chronological Train/Test Split
- Test Month: May 2026
- Training Set: X months preceding test month
- X is tunable

In [ ]:
#split data into train/test sets

def chrono_split(data, test_month=TEST_MONTH, window=TRAIN_MONTHS, date=DATE):

      months = data[date].dt.to_period("M")
      test_p = pd.Period(test_month, freq="M")
      train_start = test_p - window 

      test = data[months == test_p]
      train = data[(months >= train_start) & (months < test_p)]
      # Error checks
      assert len(train) > 0 and len(test) > 0, "ERROR: Empty split (check params)"
      assert train[date].max() < test[date].min(), "Chronology violated"
      return train, test


train_df, test_df = chrono_split(df)

print(f"Train: {train_df.shape[0]:,} rows "f"({train_df[DATE].min().date()} -> {train_df[DATE].max().date()})")
print(f"Test:  {test_df.shape[0]:,} rows "f"({test_df[DATE].min().date()} -> {test_df[DATE].max().date()})")

## Market Drift Check

- quantifying the difference between markets across the various time frames of the input csv (December to May)

In [ ]:
monthly = df.groupby(df[DATE].dt.to_period("M"))[TARGET].agg(["median", "count"])
monthly["median"].plot(marker="o", figsize=(7, 4), title="Median ClosePrice by month")

Interpret:

- The graph visualizes the differences in ClosePrice median that takes place between the months from December to May.
- As spring approaches, the median price of the single family residence properties also became more expensive
- The lowest ClosePrice median was documented during January (2026)
- The Testing month for this task is May 2026 which is also the most expensive month of the preceding months
- Training uses preceding months that are cheaper than Testing Month

The comparison of median between the preceding months to the testing month reveal that the residuals to be perform will probably under predict May, with a likely positive skew.
 

## Feature Selection
- determining a reference category
- exclusions (inclusions list is way too long)
    - date columns
    - target variable (and variations of) since feature must be knowable at query
    - CountyOrParish to avoid double counts
    - Raw variables where the log version is utilized for model
    - PostalCode 



In [ ]:
# determining a reference county based on number of listings in the cleaned csv
df["CountyOrParish"].value_counts().head(10)

LA county based on sales is the largest county and will be set as a good reference, thus is dropped.

In [ ]:
EXCLUDE = [
    "ListingKey",              
    "CloseDate", "CloseMonth",  

    "ClosePrice",               # target var
    "ClosePrice_log",           # target in log form
    "ClosePrice_repaired",      # metadata

    "CountyOrParish",           # one-hots from 02_preprocessing cleaned csv
    "PostalCode",               

    "LivingArea",               # using LivingArea_log
    "LotSizeAcres",             # using LotSizeAcres_log

    "County_Los_Angeles",       # reference

]

feature = [c for c in df.columns if c not in EXCLUDE]
print(f"{len(feature)} features")

# Checks
assert not any("ClosePrice" in c for c in feature), ("Target column leaked")
assert df[feature].isna().sum().sum() == 0, ("Feature matrix contains NaNs")

## Scaling
- spliting into x/y
- standardizing to fit on training data to make coeffs comparable

In [ ]:
#X
X_train = train_df[feature]
X_test = test_df[feature]

#
y_train = train_df[TARGET]
y_test = test_df[TARGET]

# Scale
scaler = StandardScaler()
X_train_s = pd.DataFrame(scaler.fit_transform(X_train), columns=feature, index=X_train.index)
X_test_s = pd.DataFrame(scaler.transform(X_test), columns=feature, index=X_test.index)

X_train_s.describe().loc[["mean", "std"]].T.head(8)

## Baseline Model

Testing predictions on ClosePrice in both raw form and log form for more thorough visualization. Also reverifies the EDA result in which log of the target variable is better for modeling.

- predict() plugs in the property feature values

In [ ]:
# baseline variants

# ClosePrice (raw)
model_raw = LinearRegression()
model_raw.fit(X_train_s, y_train)
pred_raw_train = model_raw.predict(X_train_s)
pred_raw_test = model_raw.predict(X_test_s)

# log(ClosePrice)
model_log = LinearRegression()
model_log.fit(X_train_s, np.log(y_train))
pred_log_train = np.exp(model_log.predict(X_train_s))
pred_log_test = np.exp(model_log.predict(X_test_s))

#check
print("Both models fitted.")

# comparing prediction distributions vs the actual (test set)
fig, ax = plt.subplots(figsize=(9, 4.5))
bins = np.linspace(0, 3_000_000, 80)   # focus on the bulk; tail clipped for readability
ax.hist(y_test, bins=bins, alpha=0.45, label="Actual", density=True)
ax.hist(pred_raw_test, bins=bins, alpha=0.45, label="raw ClosePrice", density=True)
ax.hist(pred_log_test, bins=bins, alpha=0.45, label="log(ClosePrice)", density=True)
ax.set_xlabel("ClosePrice ($)")
ax.set_ylabel("Density")
ax.set_title("Prediction distributions vs actual — test month")
ax.legend()
plt.tight_layout()
plt.show()

#check for impossible floors
print("Checking for impossible floors in predictions:")
print(f"Raw model min prediction: ${pred_raw_test.min():,.0f}")
print(f"Log model min prediction: ${pred_log_test.min():,.0f}")

### Interpretation:

- Graph:
    - The raw value distribution is very flat compared to the actual distribution 
    - The log value is much more similar to the actual distribution
- Check for impossible values:
    - The raw model predicts a negative value (−$821,582) which means the seller is paying the buyer for the property (impossible).
    - The log model predicts a positive value (will always have a positive), which is realistic ($119,294).

As expected from the EDA, the log model of ClosePrice is be the baseline target variable to be used.

## Evaluate 
- R^2 in dollar predictions for both models of raw and log ClosePrice
- using median style estimates
- training R^2 on benchmarks against the mean of training prices

In [ ]:
results = pd.DataFrame({
    "target": ["raw ClosePrice", "log ClosePrice"],
    "train_r2": [r2_score(y_train, pred_raw_train), r2_score(y_train, pred_log_train)],
    "test_r2": [r2_score(y_test, pred_raw_test), r2_score(y_test, pred_log_test)],
})

results["train_test_gap"] = results["train_r2"] - results["test_r2"]
results

### Interpret:
- log of ClosePrice yields stronger results on training predictions (again confirms that we should use the log variation)
- testing result reveals the modest behind using county as a location signal (can be refined later on)
- log has a larger train/test gap since the train fit is higher

- R_2 unchanged after excluding the reference county (LA county) which confirms that the dummy was redundant

## Coefficient Interpretation

- standardize coefficients then sorted by dollar effects

In [ ]:
coefs = pd.DataFrame({
    "feature": feature,
    "$_change_per_SD_r_model": model_raw.coef_,
    "%_change_per_SD_lg_model": model_log.coef_ * 100,
})

coefs["abs_raw"] = coefs["$_change_per_SD_r_model"].abs()
coefs = coefs.sort_values("abs_raw", ascending=False).drop(columns="abs_raw")

coefs = coefs.set_index("feature")
coefs.head(15) 

### Interpret:

- Two strongest drivers split by column:
    - LivingArea_log: strongest percent driver (+29.5%/SD) (market-wide effect)
    - BathroomsTotalInteger: strongest dollar driver (+$650k/SD) 
        - raw column is weighted toward expensive homes (squared dollar errors), where bath counts scale hard
- Size trio (bath/bed/living) is intercorrelated; shares credit and story should be read jointly
- County coeffs vs LA county reference: 
    - Bay Area premium (Santa Clara +$161k, San Mateo +$146k) 
    - Inland Empire discount (Riverside −$259k, San Bernardino −$199k)
- BedroomsTotal negative (-$214k/SD):
    - a partial result 
    - square footage cut into more bedrooms which the area already carries the value
- AttachedGarageYN negative (-$128k):
    - conditional result
    - homes lacking attached garages skew older/urban/luxury (detached garages)
- Per-SD dummy ranking favors high-vol counties while small-SD rare-county dummies shrink
    - county rows directional only

## Residual Diagnostics

- creating residual plots on both the raw and log models on the test set
- Predicted vs Actual
- checking heteroscedasticity
- failure-analysis

In [ ]:
#plotting

def diag_plots(y_true, y_pred, label):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

    #plot predicted vs actual
    axes[0].scatter(y_true, y_pred, s=6, alpha=0.3)
    lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
    axes[0].plot(lims, lims, "r--", lw=1)
    axes[0].set_xlabel("Actual ClosePrice")
    axes[0].set_ylabel("Predicted ClosePrice")
    axes[0].set_title(f"Predicted vs actual: {label}")

    #plot residuals vs predicted
    resid = y_true - y_pred
    axes[1].scatter(y_pred, resid, s=6, alpha=0.3)
    axes[1].axhline(0, color="r", ls="--", lw=1)
    axes[1].set_xlabel("Predicted ClosePrice")
    axes[1].set_ylabel("Residual (actual - predicted)")
    axes[1].set_title(f"Residuals vs predicted: {label}")

    plt.tight_layout()
    plt.show()


(y_test, pred_raw_test, "raw target (test)")
diag_plots(y_test, pred_log_test, "log target (test)")

# Zoomed in plots: the bulk of the market (< $5M)
mask = y_test < 5_000_000
diag_plots(y_test[mask], pred_raw_test[mask], "raw target (test, <$5M)")
diag_plots(y_test[mask], pred_log_test[mask], "log target (test, <$5M)")
print(f"Rows shown: {mask.sum():,} of {len(y_test):,}")
print(f"Median residual, raw: ${(y_test - pred_raw_test).median():,.0f}")
print(f"Median residual, log: ${(y_test - pred_log_test).median():,.0f}")

In [ ]:
# median residuals by month for log model

train_resid_log = y_train - pred_log_train
by_month = train_resid_log.groupby(train_df[DATE].dt.to_period("M")).median()
print(by_month)

In [ ]:
# Week 3 carry-over audit: repaired ClosePrice rows in the test month May

repaired_test = test_df[test_df["ClosePrice_repaired"] == 1]
print(f"{len(repaired_test)} repaired rows in test month")
if len(repaired_test):
    audit = repaired_test[[TARGET]].copy()
    # pred_log_test is a positionless numpy array aligned to test_df's row order;
    # translate df labels -> positions within test_df before indexing into it
    pos = test_df.index.get_indexer(repaired_test.index)
    audit["pred_log"] = pred_log_test[pos]
    audit["residual"] = audit[TARGET] - audit["pred_log"]
    print(audit)

In [ ]:
# Verdict on the $100M dot
test_df.nlargest(5, TARGET)[["ClosePrice", "LivingArea", "BedroomsTotal", "BathroomsTotalInteger", "CountyOrParish"]]

### Interpret:

- Had to zoom in and clip some of the data points since the first set of the Predicted vs Actual and Residuals vs Predicted is extremely skewed and concentrated
- log result plots show predictions tracking closely while raw model spreads at the lwoer end. 
- By-month training residuals rise January to April (−$25.2k → +$3.8k = ~$32k growth span): 
    - Test-month median residual (+$3.0k) consistent in sign, small in magnitude


## Training Window Sweep

- Re-runs the full split -> scale -> fit -> score pipeline for X 
- where X is the preceding months
- Test month remains May

In [ ]:
sweep = []
for x in range(1, 6):
    tr, te = chrono_split(df, window=x)
    Xtr, Xte = tr[feature], te[feature]
    ytr, yte = tr[TARGET], te[TARGET]

    sc = StandardScaler().fit(Xtr)
    Xtr_s, Xte_s = sc.transform(Xtr), sc.transform(Xte)

    m_raw = LinearRegression().fit(Xtr_s, ytr)
    m_log = LinearRegression().fit(Xtr_s, np.log(ytr))

    sweep.append({
        "X_months": x,
        "n_train": len(tr),
        "test_r2_raw": r2_score(yte, m_raw.predict(Xte_s)),
        "test_r2_log": r2_score(yte, np.exp(m_log.predict(Xte_s))),
    })

sweep = pd.DataFrame(sweep)
sweep

In [ ]:
#visual

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(sweep["X_months"], sweep["test_r2_raw"], "o-", label="raw target")
ax.plot(sweep["X_months"], sweep["test_r2_log"], "s-", label="log target")
ax.set_xlabel("Training window X (months)")
ax.set_ylabel("Test R^2 (dollar space)")
ax.set_title("Baseline test R^2 vs training window length")
ax.set_xticks(sweep["X_months"])
ax.legend()
plt.tight_layout()
plt.show()

### Interpret:

- X = 5 is a tunable value but here I kept it at X=5 to encompass the 6 months involved in the data set
       

## Results

- Consolidated Week 4 results — assembles previously computed objects only

In [ ]:
print("BASELINE: RESULTS")

print(f"\nData:\ncleaned CRMLS extract, {len(df):,} rows " 
      f"({df[DATE].min().date()} -> {df[DATE].max().date()})")
print(f"Split:\ntrain {len(train_df):,} rows ({TRAIN_MONTHS} months) | "
      f"test {len(test_df):,} rows ({TEST_MONTH})")
print(f"Features: {len(feature)}")

print("\nR^2 (dollar space)")
display(results)

print("Training-window sweep (test R^2, May (Test Month))")
display(sweep)

# prep for week 5
print(f"Target: log(ClosePrice), evaluated in dollar space")
print(f"Training window: X = {TRAIN_MONTHS} months")
print(f"Model to beat: test R^2 = {results.loc[1, 'test_r2']:.4f}")

## Summary / Findings

- Linear regression baseline on 66 features, chronological split (5 train months → May test)
- log(ClosePrice) is the winning target: test R^2 0.5071 vs 0.4685 raw, evaluated in dollar space
- Two independent reasons to prefer log going forward:
    - closer test fit, especially in the sub-$5M bulk of the market
    - respects the price floor
        - raw model predicts impossible negatives (min −$821k); log floor is $119k
- log(ClosePrice), X = 5 months is the Week 5 default configuration; [test R² 0.5071 is the number to beat]

Data-quality finding

- Residual analysis surfaced the two largest test-set errors as data artifacts, not model failures:
    - two in-bounds unit errors in ClosePrice (×100 and ×10, verified against raw CRMLS ListPrice)
- Week 3's bounds-only screen could not see them (both sat under the $100M ceiling)
    - fixed in 02_preprocessing with a $/sqft plausibility screen + extended factor set, producing a new cleaned .csv for this notebook to pull from
- Test R^2 rose ~0.17 (0.33 → 0.51 log) from repairing 2 of 12,007 test rows
- This is failure analysis feeding back into preprocessing 
    - the Week 3 repair machinery handled both rows cleanly once they entered its inspection set

Training-window sweep (X tunable per task doc)

- X = 1 to 5 months swept, fixed May test: 
    - log spread is 0.008 total 
    - window length is effectively irrelevant for this baseline
    - raw rises mildly with volume (0.456 → 0.469)
- 4× more training data ≈ no gain → the baseline is bias-limited, not data-limited:
    - errors come from what a linear model can't express

Next Steps:

- Week 5 will continute to build on this model